# Models

Evaluating a model involves the use of the following four functions:

  * `prepare_observation_for_inference(obs)`: This does the following -
    - Covnert numpy to pytorch
    - Adds an extra batch dim
    - Transposes the images to channel-first format
    - Rescales the images to [0, 1]
    - Adds a str field called `task`
    - Adds a str field called `robot_type`

  * `model_preproc(obs)`: Standardizes all the tensors (x - mu) / sigma including the image tensors (they didn't need rescaling after all). Adds a bunch of fields needed for RL - `action`, `next.reward`, `next.done`, `next.truncated`, `info`. Strangely enough gets rid of the `robot_type` field and converts the `task` field into a list of str. To get the standardization stats, i.e., the mean and the standard deviation, it looks at a bunch of JSON and safetensor files that are created and stored in the same place as the model weights itself during training. I can find these files in the HF gitrepo at https://huggingface.co/avilay/pick_and_place_v1.0.0/tree/main and locally on `~/.cache/huggingface/hub/models--avilay--pick_and_place_v1.0.0/`.

  * `model_postproc(action)`: Un-standardize the action values outputted by the model so they are back on their original scale.

  * `make_robot_action(action)`: Converts the action value tensor into the robot's schema of name/value pairs.

This is code from `//lerobot/scripts/lerobot_record.py::record_loop` for the evaluation step. `dataset = None`, `teleop = None`, and `policy` exists.

```python
obs = robot.get_observation()

# This is a noop
obs_processed = robot_observation_processor(obs)

observation_frame = build_dataset_frame(dataset.features, obs_processed, prefix=OBS_STR)

# action_values = predict_action(
#     observation=observation_frame,
#     policy=policy,
#     device=get_safe_torch_device(policy.config.device),
#     preprocessor=preprocessor,
#     postprocessor=postprocessor,
#     use_amp=policy.config.use_amp,
#     task=single_task,
#     robot_type=robot.robot_type,
# )

# Inside predict_action
with torch.inference_mode():
    # Converts everything to pytorch, adds a batch dimension, rescales images to [0, 1]
    observation = prepare_observation_for_inference(observation_frame, device, task, robot_type)

    # Standardizes all the tensors (x - mu) / sigma
    observation = preprocessor(observation)

    action = policy.select_action(observation)

    # Un-standardizes the output (y * sigma) + mu
    action = postprocessor(action)
# End predict_action

# Converts pytorch tensor into individual values with names
act_processed_policy: RobotAction = make_robot_action(action_values, dataset.features)

# This is noop
action_values = act_processed_policy
robot_action_to_send = robot_action_processor((act_processed_policy, obs))

_sent_action = robot.send_action(robot_action_to_send)
```

See `//scripts/eval.py` for a super simplified eval loop.

In [ ]:
import pickle
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch as th
from haikunator import Haikunator

from lerobot.policies.factory import make_pre_post_processors
from lerobot.datasets.utils import build_dataset_frame, hw_to_dataset_features
from lerobot.policies.utils import prepare_observation_for_inference, make_robot_action
from lerobot.robots.so_follower import SO101Follower, SO101FollowerConfig
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.utils.constants import OBS_STR
from lerobot.policies.act.modeling_act import ACTPolicy

In [ ]:
robot_cfg = SO101FollowerConfig(
    port="/dev/ttyACM2",
    id="yantra_robot",
    cameras={
        "gripper": OpenCVCameraConfig(
            index_or_path=Path("/dev/video51"),
            width=640,
            height=480,
            fps=30,
            warmup_s=3,
        ),
        "env": OpenCVCameraConfig(
            index_or_path=Path("/dev/video48"),
            width=640,
            height=480,
            fps=30,
            warmup_s=3,
        ),
    },
)
robot = SO101Follower(robot_cfg)

In [ ]:
obs_pkl = Path("obs.pkl")
if not obs_pkl.exists():
    robot.connect(calibrate=False)
    obs = robot.get_observation()
    with open(obs_pkl, "wb") as fout:
        pickle.dump(obs, fout)
    robot.disconnect()
else:
    with open("obs.pkl", "rb") as fin:
        obs = pickle.load(fin)

for key, value in obs.items():
    if isinstance(value, np.ndarray):
        print(
            f"[{key}] type: {type(value)}, shape: {value.shape}, dtype: {value.dtype}"
        )
        print(
            f"\tstats: min={np.min(value)}, mean={np.mean(value)}, max={np.max(value)}, std={np.std(value)}"
        )
    else:
        print(f"[{key}] type: {type(value)}, value: {value}")

[shoulder_pan.pos] type: <class 'float'>, value: 0.6593406593406593
[shoulder_lift.pos] type: <class 'float'>, value: -105.18681318681318
[elbow_flex.pos] type: <class 'float'>, value: 98.1978021978022
[wrist_flex.pos] type: <class 'float'>, value: 59.16483516483517
[wrist_roll.pos] type: <class 'float'>, value: -3.208791208791209
[gripper.pos] type: <class 'float'>, value: 0.6816632583503749
[gripper] type: <class 'numpy.ndarray'>, shape: (480, 640, 3), dtype: uint8
	stats: min=4, mean=133.28033637152777, max=255, std=56.79012018383561
[env] type: <class 'numpy.ndarray'>, shape: (480, 640, 3), dtype: uint8
	stats: min=0, mean=135.91257486979165, max=255, std=46.30363506339924


In [ ]:
action_features = hw_to_dataset_features(robot.action_features, "action")  # type: ignore
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}

In [ ]:
obs_frame = build_dataset_frame(dataset_features, obs, OBS_STR)
for key, value in obs_frame.items():
    tvalue = "...array..." if key.find("images") > -1 else value
    print(
        f"[{key}] type: {type(value)}, shape: {value.shape}, dtype: {value.dtype}, value: {tvalue}"
    )
    print(
        f"\tstats: min={np.min(value)}, mean={np.mean(value)}, max={np.max(value)}, std={np.std(value)}"
    )

[observation.state] type: <class 'numpy.ndarray'>, shape: (6,), dtype: float32, value: [   0.6593407 -105.18681     98.1978      59.164837    -3.2087913
    0.6816633]
	stats: min=-105.18681335449219, mean=8.384673118591309, max=98.19779968261719, std=62.9774169921875
[observation.images.gripper] type: <class 'numpy.ndarray'>, shape: (480, 640, 3), dtype: uint8, value: ...array...
	stats: min=4, mean=133.28033637152777, max=255, std=56.79012018383561
[observation.images.env] type: <class 'numpy.ndarray'>, shape: (480, 640, 3), dtype: uint8, value: ...array...
	stats: min=0, mean=135.91257486979165, max=255, std=46.30363506339924


In [10]:
cpu = th.device("cpu")
task = Haikunator().haikunate()
task

'frosty-base-6843'

In [ ]:
# This call will mutate obs_frame, so lets make a copy
# lerobot code makes a shallow copy, I wonder why?
# Does the following:
#   - Covnert numpy to pytorch
#   - Adds an extra batch dim
#   - Transposes the images to channel-first format
#   - Rescales the images to [0, 1]
#   - Adds a str field called task
#   - Adds a str field called robot_type
obs_frame_copy = deepcopy(obs_frame)
obs_input = prepare_observation_for_inference(
    obs_frame_copy, cpu, robot_type=robot.robot_type
)
for key, value in obs_input.items():
    if isinstance(value, th.Tensor):
        tvalue = "...array..." if key.find("images") > -1 else value
        print(
            f"[{key}] type: {type(value)}, shape: {value.shape}, dtype: {value.dtype}, value: {tvalue}"
        )
        print(
            f"\tstats: min={th.min(value)}, mean={th.mean(value)}, max={th.max(value)}, std={th.std(value)}"
        )
    else:
        print(f"[{key}] type: {type(value)}, value: {value}")

[observation.state] type: <class 'torch.Tensor'>, shape: torch.Size([1, 6]), dtype: torch.float32, value: tensor([[   0.6593, -105.1868,   98.1978,   59.1648,   -3.2088,    0.6817]])
	stats: min=-105.18681335449219, mean=8.384672164916992, max=98.19779968261719, std=68.9883041381836
[observation.images.gripper] type: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 480, 640]), dtype: torch.float32, value: ...array...
	stats: min=0.01568627543747425, mean=0.5226680040359497, max=1.0, std=0.22270648181438446
[observation.images.env] type: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 480, 640]), dtype: torch.float32, value: ...array...
	stats: min=0.0, mean=0.5329904556274414, max=1.0, std=0.1815829873085022
[task] type: <class 'str'>, value: 
[robot_type] type: <class 'str'>, value: so_follower


In [ ]:
# In the //lerobot/scripts/lerobot_record.py there are a couple of function calls with the dataset as an argument.
# In the eval path, the dataset is not really needed, and indeed when I look at the code, dataset.meta.stats is only used
# in the training path, and dataset.features only uses the ["action"] section which is identical to the dataset_features["action"]

from lerobot.datasets.lerobot_dataset import LeRobotDataset

task = Haikunator().haikunate()
dataset = LeRobotDataset.create(
    repo_id=f"avilay/{task}",
    fps=30,
    features=dataset_features,
    robot_type=robot.robot_type,
)

assert dataset.meta.stats is None
dataset.features["action"] == dataset_features["action"]

True

In [14]:
model_path = "avilay/pick_and_place_v1.0.0"
policy = ACTPolicy.from_pretrained(model_path)
policy

ACTPolicy(
  (model): ACT(
    (vae_encoder): ACTEncoder(
      (layers): ModuleList(
        (0-3): 4 x ACTEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=3200, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=3200, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): Identity()
    )
    (vae_encoder_cls_embed): Embedding(1, 512)
    (vae_encoder_robot_state_input_proj): Linear(in_features=6, out_features=512, bias=True)
    (vae_encoder_action_input_proj): Linear(in_features=6, out_features=512, bias

In [15]:
# In the documentation sample code as well as in //lerobot/scripts/lerobot_record.py::record they pass in the dataset.meta.stats
# In the eval path this is not used and is anyway empty for an empty dataset.
model_preproc, model_postproc = make_pre_post_processors(
    policy_cfg=policy,  # type: ignore
    # dataset_stats=dataset.meta.stats
    pretrained_path=model_path,
    preprocessor_overrides={"device_processor": {"device": "cpu"}},
    postprocessor_overrides={"device_processor": {"device": "cpu"}},
)

In [ ]:
# Standardizes all the tensors (x - mu) / sigma including the image tensors (they didn't need rescaling after all)
# Adds a bunch of fields needed for RL - action, next.reward, next.done, next.truncated, info
# Strangely enough gets rid of the robot_type field and converts the task field into a list of str
obs_input_processed = model_preproc(obs_input)
for key, value in obs_input_processed.items():
    if isinstance(value, th.Tensor):
        tvalue = "...array..." if key.find("images") > -1 else value
        print(
            f"[{key}] type: {type(value)}, shape: {value.shape}, dtype: {value.dtype}, value: {tvalue}"
        )
        print(
            f"\tstats: min={th.min(value)}, mean={th.mean(value)}, max={th.max(value)}, std={th.std(value)}"
        )
    else:
        print(f"[{key}] type: {type(value)}, value: {value}")

[action] type: <class 'NoneType'>, value: None
[next.reward] type: <class 'float'>, value: 0.0
[next.done] type: <class 'bool'>, value: False
[next.truncated] type: <class 'bool'>, value: False
[info] type: <class 'dict'>, value: {}
[task] type: <class 'list'>, value: ['frosty-base-6843']
[observation.state] type: <class 'torch.Tensor'>, shape: torch.Size([1, 6]), dtype: torch.float32, value: tensor([[-0.3295, -1.4657,  1.3103, -1.4127, -2.1106, -0.7372]])
	stats: min=-2.110630989074707, mean=-0.7909041047096252, max=1.3102872371673584, std=1.2012088298797607
[observation.images.gripper] type: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 480, 640]), dtype: torch.float32, value: ...array...
	stats: min=-2.0494048595428467, mean=0.3265286982059479, max=2.6399998664855957, std=0.9757128953933716
[observation.images.env] type: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 480, 640]), dtype: torch.float32, value: ...array...
	stats: min=-1.8610326051712036, mean=0.3716713786125183, m

In [ ]:
# Outputs the standardized action values
action = policy.select_action(obs_input_processed)
print(
    f"type: {type(action)}, shape: {action.shape}, dtype: {action.dtype}\nvalue: {action}"
)

type: <class 'torch.Tensor'>, shape: torch.Size([1, 6]), dtype: torch.float32
value: tensor([[-0.3334, -1.1424,  1.1076, -1.1729, -2.3683, -0.5117]])


In [ ]:
# Un-standardize the action values so they are back on their original scale
action_processed = model_postproc(action)
print(
    f"type: {type(action_processed)}, shape: {action_processed.shape}, dtype: {action_processed.dtype}\nvalue: {action_processed}"
)

type: <class 'torch.Tensor'>, shape: torch.Size([1, 6]), dtype: torch.float32
value: tensor([[  0.5580, -88.3210,  86.6901,  62.0366,  -4.2347,   2.1967]])


In [ ]:
# In lerobot_record.py the second arg is dataset.features which has a bunch of default fields in addition to dataset_features
# However in the make_robot_action it only uses dataset.features["action"] which is identical to dataset_features["action"]
# So I can safely pass in dataset_features here.
# Converts the action value tensor into the robot's schema
robot_action = make_robot_action(action_processed, dataset_features)
robot_action

{'shoulder_pan.pos': 0.5580167770385742,
 'shoulder_lift.pos': -88.32095336914062,
 'elbow_flex.pos': 86.69007873535156,
 'wrist_flex.pos': 62.03657531738281,
 'wrist_roll.pos': -4.234726905822754,
 'gripper.pos': 2.1966514587402344}